# User Journey Drop-Off Analysis

## Project Overview
Analyze event-level product journey data to identify where users exit a multi-step journey and what segments experience the largest drop-off. This helps product teams pinpoint friction points and prioritize UX improvements.

## Learning Objectives
- Build conversion funnels from event-level data
- Calculate step-to-step and overall conversion rates
- Segment drop-off patterns by user type, device, and acquisition channel
- Identify the highest-impact friction points

## Problem Statement
Users start a key product journey (e.g., signup → onboarding → first action → habit formation) but many drop off before completing it. The product team needs to know *where* and *who* drops off most to prioritize improvements.

## Why This Project Matters
Every 1% improvement in conversion at a high-drop-off step can translate directly to revenue and user growth. Journey analysis is how companies like Spotify, Uber, and Shopify optimize their core user flows.

## Dataset Overview
Synthetic event-level data: ~8K users going through a 6-step journey. Each event has a timestamp, user ID, step name, device type, and acquisition channel. Not all users complete all steps.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- Patterns inspired by SaaS/e-commerce onboarding funnels
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))

JOURNEY_STEPS = ['Visit Landing Page', 'Sign Up', 'Complete Profile',
                 'First Action', 'Return Visit', 'Become Regular']
print(f'Journey steps: {JOURNEY_STEPS}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n_users = 8000

devices = ['Desktop', 'Mobile', 'Tablet']
device_weights = [0.45, 0.40, 0.15]

channels = ['Organic Search', 'Paid Ads', 'Referral', 'Social Media', 'Direct']
channel_weights = [0.30, 0.25, 0.15, 0.20, 0.10]

# Step-to-step conversion rates (vary by device)
base_conv = [1.0, 0.55, 0.70, 0.60, 0.50, 0.65]  # conditional on reaching previous step
device_modifier = {'Desktop': 1.0, 'Mobile': 0.85, 'Tablet': 0.90}
channel_modifier = {'Organic Search': 1.05, 'Paid Ads': 0.90, 'Referral': 1.10,
                     'Social Media': 0.85, 'Direct': 1.0}

rows = []
base_time = pd.Timestamp('2024-01-01')

for uid in range(n_users):
    device = np.random.choice(devices, p=device_weights)
    channel = np.random.choice(channels, p=channel_weights)
    d_mod = device_modifier[device]
    c_mod = channel_modifier[channel]

    time_offset = pd.Timedelta(hours=np.random.randint(0, 24*365))

    for step_idx, step in enumerate(JOURNEY_STEPS):
        conv_rate = base_conv[step_idx] * d_mod * c_mod
        if step_idx == 0 or np.random.random() < conv_rate:
            rows.append({
                'UserID': f'U{uid:06d}',
                'Step': step,
                'StepOrder': step_idx + 1,
                'Device': device,
                'Channel': channel,
                'Timestamp': base_time + time_offset + pd.Timedelta(minutes=step_idx * np.random.randint(5, 120)),
            })
        else:
            break  # user dropped off

df = pd.DataFrame(rows)
df = df.sort_values(['UserID', 'StepOrder']).reset_index(drop=True)
print(f'Dataset shape: {df.shape}')
print(f'Unique users: {df["UserID"].nunique()}')
df.head(12)

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Users per step:')
step_counts = df.groupby('Step')['UserID'].nunique().reindex(JOURNEY_STEPS)
print(step_counts)
print(f'\nDevices: {df["Device"].value_counts().to_dict()}')
print(f'Channels: {df["Channel"].value_counts().to_dict()}')

## Exploratory Data Analysis — Overall Funnel

In [ ]:
funnel = step_counts.values
funnel_pct = funnel / funnel[0] * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Absolute funnel
axes[0].barh(JOURNEY_STEPS[::-1], funnel[::-1], edgecolor='black',
             color=plt.cm.Blues(np.linspace(0.3, 0.9, len(JOURNEY_STEPS))))
axes[0].set_title('User Journey Funnel (Absolute)')
axes[0].set_xlabel('Users')
for i, (v, pct) in enumerate(zip(funnel[::-1], funnel_pct[::-1])):
    axes[0].text(v + 50, i, f'{v:,} ({pct:.0f}%)', va='center')

# Step-to-step conversion
step_conv = [100.0]
for i in range(1, len(funnel)):
    step_conv.append(funnel[i] / funnel[i-1] * 100 if funnel[i-1] > 0 else 0)

axes[1].bar(range(len(JOURNEY_STEPS)), step_conv, edgecolor='black', color='coral', alpha=0.8)
axes[1].set_xticks(range(len(JOURNEY_STEPS)))
axes[1].set_xticklabels([s.replace(' ', '\n') for s in JOURNEY_STEPS], fontsize=8)
axes[1].set_title('Step-to-Step Conversion Rate')
axes[1].set_ylabel('Conversion %')
axes[1].set_ylim(0, 110)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'overall_funnel.png'), dpi=100, bbox_inches='tight')
plt.show()

print('Step-to-step conversion:')
for step, conv in zip(JOURNEY_STEPS, step_conv):
    print(f'  {step}: {conv:.1f}%')

## Drop-Off Analysis — Where Do Users Leave?

In [ ]:
# Find last step per user
last_step = df.groupby('UserID')['StepOrder'].max().reset_index()
last_step.columns = ['UserID', 'LastStep']
last_step['DropOffStep'] = last_step['LastStep'].map(dict(enumerate(JOURNEY_STEPS, 1)))

dropoff_dist = last_step['DropOffStep'].value_counts().reindex(JOURNEY_STEPS)
print('Users whose journey ended at each step:')
print(dropoff_dist)

fig, ax = plt.subplots(figsize=(10, 5))
dropoff_dist.plot.bar(ax=ax, edgecolor='black', color='salmon')
ax.set_title('Where Users Drop Off (Last Step Reached)')
ax.set_ylabel('Users')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'dropoff_distribution.png'), dpi=100, bbox_inches='tight')
plt.show()

## Drop-Off by Device

In [ ]:
device_funnels = {}
for device in devices:
    device_df = df[df['Device'] == device]
    counts = device_df.groupby('Step')['UserID'].nunique().reindex(JOURNEY_STEPS).fillna(0)
    device_funnels[device] = (counts / counts.iloc[0] * 100).values

device_funnel_df = pd.DataFrame(device_funnels, index=JOURNEY_STEPS)

fig, ax = plt.subplots(figsize=(12, 6))
device_funnel_df.plot(ax=ax, marker='o', linewidth=2)
ax.set_title('Conversion Funnel by Device')
ax.set_ylabel('% of Initial Users')
ax.set_xlabel('Journey Step')
ax.legend(title='Device')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'funnel_by_device.png'), dpi=100, bbox_inches='tight')
plt.show()

## Drop-Off by Acquisition Channel

In [ ]:
channel_funnels = {}
for ch in channels:
    ch_df = df[df['Channel'] == ch]
    counts = ch_df.groupby('Step')['UserID'].nunique().reindex(JOURNEY_STEPS).fillna(0)
    channel_funnels[ch] = (counts / counts.iloc[0] * 100).values

channel_funnel_df = pd.DataFrame(channel_funnels, index=JOURNEY_STEPS)

fig, ax = plt.subplots(figsize=(12, 6))
channel_funnel_df.plot(ax=ax, marker='o', linewidth=2)
ax.set_title('Conversion Funnel by Acquisition Channel')
ax.set_ylabel('% of Initial Users')
ax.legend(title='Channel', bbox_to_anchor=(1.05, 1))
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'funnel_by_channel.png'), dpi=100, bbox_inches='tight')
plt.show()

## Biggest Drop-Off Point by Segment

In [ ]:
print('Biggest step-to-step drop-off by segment:')
print('=' * 60)

# Overall
for i in range(1, len(JOURNEY_STEPS)):
    drop = funnel_pct[i-1] - funnel_pct[i]
    if drop == max(funnel_pct[j-1] - funnel_pct[j] for j in range(1, len(JOURNEY_STEPS))):
        print(f'  Overall: {JOURNEY_STEPS[i-1]} → {JOURNEY_STEPS[i]} (drop: {drop:.1f}%)')

# By device
for device in devices:
    f = device_funnel_df[device].values
    drops = [f[i-1] - f[i] for i in range(1, len(f))]
    max_idx = np.argmax(drops)
    print(f'  {device}: {JOURNEY_STEPS[max_idx]} → {JOURNEY_STEPS[max_idx+1]} (drop: {drops[max_idx]:.1f}%)')

# By channel
for ch in channels:
    f = channel_funnel_df[ch].values
    drops = [f[i-1] - f[i] for i in range(1, len(f))]
    max_idx = np.argmax(drops)
    print(f'  {ch}: {JOURNEY_STEPS[max_idx]} → {JOURNEY_STEPS[max_idx+1]} (drop: {drops[max_idx]:.1f}%)')

## Completion Rate Summary

In [ ]:
completion_rate = funnel[-1] / funnel[0] * 100
print(f'Overall journey completion rate: {completion_rate:.1f}%')
print(f'\nCompletion rate by device:')
for device in devices:
    print(f'  {device}: {device_funnel_df[device].iloc[-1]:.1f}%')
print(f'\nCompletion rate by channel:')
for ch in channels:
    print(f'  {ch}: {channel_funnel_df[ch].iloc[-1]:.1f}%')

## Interpretation and Business Insight
- The **Sign Up** step is the biggest drop-off point overall — nearly half of visitors don't convert
- **Mobile** users drop off more at every step compared to Desktop, especially at Complete Profile — mobile forms may need simplification
- **Social Media** and **Paid Ads** visitors convert at lower rates, suggesting lower intent traffic
- **Referral** users have the highest completion rate, confirming that warm introductions drive better engagement
- The **Return Visit** step shows significant attrition — users complete first actions but don't form habits

## Limitations
- Synthetic data with pre-set conversion rates — real funnels have more variability
- No time-between-steps analysis (how long users take at each step)
- Sequential funnel only — doesn't capture users who skip steps or take non-linear paths
- No A/B test data to attribute improvements to specific changes

## How to Improve This Project
- Use real event-stream data from product analytics tools
- Add time-to-convert analysis for each step
- Build cohort-based funnels to track improvements over time
- Add exit page / exit action analysis to understand what users do before leaving
- Implement statistical significance testing for segment differences

## Production Considerations
- Instrument every step with timestamped events for real-time funnel tracking
- Set up automated alerts when step conversion rates drop below thresholds
- Run A/B tests on high-drop-off steps and measure impact on funnel completion
- Segment funnels by user cohort (signup week) to track improvement over time

## Common Mistakes
- Using total events instead of unique users for funnel counts (inflates numbers)
- Not defining a time window for funnel completion (user who takes 30 days vs 1 day)
- Comparing segments without controlling for volume (small segments have noisy rates)
- Optimizing early-funnel conversion without checking if it improves end-to-end completion

## Mini Challenge / Exercises
1. What is the "expected" number of users completing the journey if you could fix the worst step to match the best step's conversion rate?
2. Calculate the median time between steps for users who complete the full journey.
3. Create a "fast completers" vs "slow completers" segment. Do they differ by device?
4. If 10,000 more users entered the funnel, how many would complete the journey at current rates?

## Final Summary / Key Takeaways
- Funnel analysis is the cornerstone of product growth and conversion optimization
- The biggest drop-off step deserves the most attention and resources
- Segmenting funnels by device, channel, and user type reveals hidden friction points
- Mobile-specific optimizations are critical given mobile's growing share
- Referral traffic consistently outperforms paid channels in conversion quality